# Predicción final

#### Dependencias

In [1]:
import pandas as pd
import numpy as np
import joblib

#### Funciones

In [2]:
def corregir_horas(df, columnas):
    for col in columnas:
        horas = df[col] * 24
        enteras = np.floor(horas + 0.000001)
        minutos = np.round((horas - enteras) * 60)
        df[col] = enteras + minutos / 10
    return df

#### Lectura

In [3]:
df_original = pd.read_excel("../data/raw/smoking_prediction_entrega.xlsx")
df_original.shape 

(5692, 26)

In [4]:
df = df_original.copy() # trabajamos sobre una copia para no tocar el original

#### Carga del pipeline

In [5]:
limites = joblib.load("../models/limites_outliers.joblib")
ohe = joblib.load("../models/ohe.joblib")
modelo = joblib.load("../models/modelo_xgb.joblib")
umbral = joblib.load("../models/umbral.joblib")
umbral 

0.38

#### Transformaciones

In [6]:
columnas_horas = df.select_dtypes(include="float").columns.tolist()
df = corregir_horas(df, columnas_horas)
df[["waist(cm)", "systolic", "hemoglobin"]].head(3)

,waist(cm),systolic,hemoglobin
0,82.5,121.0,15.9
1,83.0,129.0,14.6
2,82.0,126.0,15.1


In [7]:
X_pred = df.drop(columns=["ID", "oral"])

In [8]:
X_pred[limites.index] = X_pred[limites.index].clip(upper=limites, axis=1)

In [9]:
cat_cols = ["gender", "tartar"]
X_pred_ohe = pd.DataFrame(ohe.transform(X_pred[cat_cols]), columns=ohe.get_feature_names_out(cat_cols), index=X_pred.index)
X_pred = pd.concat([X_pred.drop(columns=cat_cols), X_pred_ohe], axis=1)

In [10]:
columnas_modelo = pd.read_csv("../data/processed/X_train.csv", nrows=0).columns
X_pred = X_pred[columnas_modelo] # mismas columnas y en el mismo orden que en el entrenamiento
X_pred.shape # (5692, 26)

(5692, 26)

Al archivo nuevo le apliqué exactamente los mismos pasos que al train: la corrección de horas, el recorte con los mismos límites y el mismo encoder. No se borró ninguna fila y quedaron las mismas 26 columnas, en el mismo orden.

#### Predicción

In [11]:
proba = modelo.predict_proba(X_pred)[:, 1]  # probabilidad de que cada persona fume
prediccion = (proba >= umbral).astype(int)  # 1 si supera el umbral de 0,38, 0 si no

In [12]:
df_original["smoking_prediction"] = prediccion
df_original["smoking_prediction"].value_counts(normalize=True) * 100 # % de fumadores predichos

smoking_prediction
1    53.109628
0    46.890372
Name: proportion, dtype: float64

In [13]:
df_original["smoking_prediction"].unique() # solo debe haber 0 y 1

array([0, 1])

In [14]:
df_original["smoking_prediction"].isnull().sum() # 0: todas las personas tienen predicción

np.int64(0)

In [15]:
df_original[["ID", "gender", "age", "smoking_prediction"]].head(10)

,ID,gender,age,smoking_prediction
0,27358,M,25,0
1,27364,M,30,1
2,27368,M,55,1
3,27378,M,20,0
4,27381,M,25,1
5,27386,M,45,1
6,27388,M,55,1
7,27393,F,50,0
8,27406,F,45,0
9,27417,M,40,1


#### Exportar

In [16]:
df_original.to_csv("../data/processed/smoking_prediction_entrega_resultado.csv", index=False)

In [17]:
pd.crosstab(df_original["gender"], df_original["smoking_prediction"], normalize="index") * 100 # % de fumadores predichos por género

smoking_prediction,0,1
gender,,
F,99.903614,0.096386
M,16.477744,83.522256


In [18]:
resultado = pd.read_csv("../data/processed/smoking_prediction_entrega_resultado.csv")
resultado.shape 

(5692, 27)

Se predijo que el 53% fuma. Casi ninguna mujer sale como fumadora y los hombres sí en su mayoría, igual que en los datos de entrenamiento. El porcentaje es más alto que el real (36%) porque usamos el umbral de 0,38, que prioriza encontrar a los fumadores.